# Data Assessment, Cleaning & Forecast-Readiness EDA (`Part Name + Region Id`)

Executes full raw data assessment, applies the 4 business cleaning rules (holding off on segment imputation—blank segments are removed), and produces post-cleaning EDA, `2022–2025` year-presence cohorts, and forecasting-readiness reports at the **`Part Name + Region Id`** modeling grain alongside a lightweight **`Bulk Value + Region Id`** WMAPE evaluation check.

### Key Updates in v5
- **Restored Full Reporting Coverage:** Brought back `4.1d Discontinued (Part Name + Region Id) by Region × Segment`, `4.2 Cross Reconciliation: Business Segment × Quarter`, `7.2b Collapsed 2023–2025 Iterations (Row-Presence Basis)`, and `7.8 Top 20 Unexplained Monthly Outlier Rows` (across notebook output, Excel tabs, and PDF export), while keeping `4.3` transaction hotspots focused on `Part-Site-Region-Segment`.
- **Quarterly Launch Cohorts (`Section 7.7`):** Quarter-level launch cohorts (`First_Pos_Quarter`: `2022-Q4` through `2025-Q4`) alongside `First_Pos_Year`, carried into `series_catalog` and `cleaned_monthly_part_region.csv`.
- **Post-Cleaning Quarterly YoY Comparison (`Section 6.5b` & `6.7`):** `Quarter (Q1–Q4) × Year (2022–2025)` post-cleaning quantity comparison table with YoY % growth plus grouped quarterly bar chart.
- **Section 3.1 Individual Two-Way Mapping Tables (`Cell 4A`):** 9 separate, plain-English 2-way mapping tables (`3.1.1`–`3.1.9`) with dedicated Excel tabs.

## Cell 1: Imports, Configuration & Strict-Dtype CSV Loader

In [ ]:
import os
import itertools
import importlib.util
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ==========================================
# CONFIGURATION
# ==========================================
FILE_PATH = "CC_History_Actuals_till_2025.csv"   # <-- Update to your CSV filename
OUTPUT_EXCEL = "Data_Assessment_and_EDA_Report.xlsx"
OUTPUT_MONTHLY_CSV = "cleaned_monthly_part_region.csv"
OUTPUT_SEC6_PNG = "Section6_Core_EDA_Bar_Pie_Charts.png"
OUTPUT_DASHBOARD_PNG = "EDA_and_Year_Iterations_Dashboard.png"
OUTPUT_PDF = "Data_Assessment_and_EDA_Report.pdf"

DAYFIRST = True                 # Used only for non-ISO date strings (e.g. 26-Oct-22 or 03/04/2025)
YEARS = [2022, 2023, 2024, 2025]
MIN_OBS_SERIES_OUTLIER = 8      # Min positive rows for per-series transaction IQR fence
MIN_OBS_MONTHLY_OUTLIER = 12    # Min positive months for per-series monthly IQR fence
SEASONAL_TOL = 1.5              # Flagged month within this factor of same-month-other-years mean => likely seasonal
T1_MIN_POS_MONTHS = 18          # Tier 1 threshold
T2_MIN_POS_MONTHS = 12          # Tier 2 threshold

EXPECTED_COLS = [
    "Date", "Part Name", "Part Description", "Part Site", "Bulk Value", "Quantity",
    "Business Segment Value", "Business Segment Description", "Region Id"
]
EXPECTED_STRING_COLS = [c for c in EXPECTED_COLS if c not in ("Date", "Quantity")]

# Load everything as string initially: preserves leading zeros and avoids silent coercion
file_size_mb = os.path.getsize(FILE_PATH) / (1024 * 1024)
raw_str_df = pd.read_csv(FILE_PATH, dtype=str, keep_default_na=False)
raw_str_df.columns = raw_str_df.columns.str.strip()

missing_cols = [c for c in EXPECTED_COLS if c not in raw_str_df.columns]
assert not missing_cols, f"Missing expected columns: {missing_cols}"

print(f"Loaded '{FILE_PATH}' | Shape: {raw_str_df.shape[0]:,} rows x {raw_str_df.shape[1]} cols | {file_size_mb:.2f} MB")
display(raw_str_df.head(5))

## Cell 2: Section 1 — Basic Data Assessment (Raw Data)

In [ ]:
# ==========================================
# 1. BASIC DATA ASSESSMENT (RAW DATA)
# ==========================================
df_raw = raw_str_df[EXPECTED_COLS].copy()

# NOTE: Bare "NA" is deliberately NOT treated as null (can represent Region Id = North America).
NULL_SENTINELS = {"", "N/A", "NULL", "NONE", "NAN", "#N/A"}
na_literals = {c: int((df_raw[c].astype(str).str.strip().str.upper() == "NA").sum()) for c in df_raw.columns}
na_literals = {c: v for c, v in na_literals.items() if v}
if na_literals:
    print(f"Literal 'NA' values kept as real category codes (not nulls): {na_literals}")

for col in df_raw.columns:
    s = df_raw[col].astype(str).str.strip()
    df_raw[col] = s.mask(s.str.upper().isin(NULL_SENTINELS))

# Date parsing: ISO first, dayfirst fallback for non-ISO strings
date_str = df_raw["Date"]
iso_parsed = pd.to_datetime(date_str, format="ISO8601", errors="coerce")
fallback_mask = iso_parsed.isna() & date_str.notna()
fallback_parsed = pd.to_datetime(date_str[fallback_mask], format="mixed", dayfirst=DAYFIRST, errors="coerce")
parsed_dates = iso_parsed.copy()
parsed_dates.loc[fallback_mask] = fallback_parsed

print(
    f"Dates parsed as ISO: {int(iso_parsed.notna().sum()):,} | "
    f"via dayfirst={DAYFIRST} fallback: {int(fallback_parsed.notna().sum()):,} | "
    f"unparseable: {int((date_str.notna() & parsed_dates.isna()).sum()):,}"
)

# Quantity parsing
qty_str = df_raw["Quantity"].str.replace(",", "", regex=False)
parsed_qty = pd.to_numeric(qty_str, errors="coerce")
qty_invalid_mask = df_raw["Quantity"].notna() & parsed_qty.isna()
qty_decimal_mask = parsed_qty.notna() & (parsed_qty % 1 != 0)
date_invalid_mask = df_raw["Date"].notna() & parsed_dates.isna()

# 1.1 Data Type Mismatch Report
type_mismatch_rows = [
    {
        "Column": "Date", "Expected Type": "Date",
        "Non-Null Count": int(df_raw["Date"].notna().sum()),
        "Invalid/Unparseable Count": int(date_invalid_mask.sum()),
        "Sample Mismatches": list(df_raw.loc[date_invalid_mask, "Date"].unique()[:5]),
    },
    {
        "Column": "Quantity", "Expected Type": "Numeric",
        "Non-Null Count": int(df_raw["Quantity"].notna().sum()),
        "Invalid/Unparseable Count": int(qty_invalid_mask.sum()),
        "Sample Mismatches": list(df_raw.loc[qty_invalid_mask, "Quantity"].unique()[:5]),
    },
]
for col in EXPECTED_STRING_COLS:
    pure_numeric_count = int(df_raw[col].dropna().str.match(r"^-?\d+(\.\d+)?$").sum())
    type_mismatch_rows.append({
        "Column": col, "Expected Type": "String",
        "Non-Null Count": int(df_raw[col].notna().sum()),
        "Invalid/Unparseable Count": 0,
        "Sample Mismatches": f"{pure_numeric_count:,} numeric-formatted strings (kept as str)",
    })
df_type_mismatch = pd.DataFrame(type_mismatch_rows)
print(f"Quantity rows with non-zero decimals (e.g. .50): {int(qty_decimal_mask.sum()):,}")
print("\n--- 1.1 DATA TYPE MISMATCH REPORT ---")
display(df_type_mismatch)

df_raw["Date"] = parsed_dates
df_raw["Quantity"] = parsed_qty
df_raw["Year"] = df_raw["Date"].dt.year.astype("Int64")
df_raw["YearMonth"] = df_raw["Date"].dt.to_period("M")
df_raw["YearWeek"] = df_raw["Date"].dt.to_period("W")

def unique_part_regions(df, by):
    d = df.dropna(subset=["Part Name", "Region Id"])
    return (
        d.drop_duplicates(list(by) + ["Part Name", "Region Id"])
        .groupby(list(by), dropna=False).size().rename("Unique_Part_Regions").reset_index()
    )

# 1.2 Column Profile
col_profile = []
for col in EXPECTED_COLS:
    null_cnt = int(df_raw[col].isna().sum())
    pop_cnt = len(df_raw) - null_cnt
    top_val = df_raw[col].mode(dropna=True).iloc[0] if pop_cnt > 0 else np.nan
    col_profile.append({
        "Column": col, "Inferred Dtype": str(df_raw[col].dtype), "Total Records": len(df_raw),
        "Populated Count": pop_cnt, "Null Count": null_cnt,
        "Null %": round(null_cnt / len(df_raw) * 100, 3),
        "Unique Values": int(df_raw[col].nunique(dropna=True)),
        "Top Value": top_val, "Top Value Freq": int((df_raw[col] == top_val).sum()) if pop_cnt > 0 else 0,
    })
df_col_profile = pd.DataFrame(col_profile)
print("\n--- 1.2 COLUMN PROFILE ---")
display(df_col_profile)

# 1.3 Master Entity Summary
master_summary = pd.DataFrame([
    {"Entity / Combination": f"Distinct {c}", "Unique Count": int(df_raw[c].nunique(dropna=True))}
    for c in [
        "Part Name", "Part Description", "Part Site", "Bulk Value",
        "Business Segment Value", "Business Segment Description", "Region Id"
    ]
] + [
    {
        "Entity / Combination": "Distinct (Part Name + Region Id) [Forecast Grain]",
        "Unique Count": len(df_raw[["Part Name", "Region Id"]].dropna().drop_duplicates()),
    },
    {
        "Entity / Combination": "Distinct (Bulk Value + Region Id) [WMAPE Eval Grain]",
        "Unique Count": len(df_raw[["Bulk Value", "Region Id"]].dropna().drop_duplicates()),
    },
    {
        "Entity / Combination": "Distinct (Part Name + Part Site + Region Id)",
        "Unique Count": len(df_raw[["Part Name", "Part Site", "Region Id"]].dropna().drop_duplicates()),
    },
])
print("\n--- 1.3 MASTER ENTITY SUMMARY ---")
display(master_summary)

# 1.4 Historical Coverage
min_date, max_date = df_raw["Date"].min(), df_raw["Date"].max()
total_cal_months = (max_date.year - min_date.year) * 12 + (max_date.month - min_date.month) + 1
print("\n--- 1.4 HISTORICAL COVERAGE ---")
print(f"Min Date: {min_date:%Y-%m-%d} | Max Date: {max_date:%Y-%m-%d} | Span: {total_cal_months} calendar months")

# 1.5 & 1.6 Grain & Key Uniqueness
key_candidates = [
    ("Full Row (all 9 columns)", EXPECTED_COLS),
    ("Daily: Date + Part Name + Part Site + Business Segment Value + Region Id", ["Date", "Part Name", "Part Site", "Business Segment Value", "Region Id"]),
    ("Daily: Date + Part Name + Part Site + Region Id", ["Date", "Part Name", "Part Site", "Region Id"]),
    ("Daily: Date + Part Name + Region Id", ["Date", "Part Name", "Region Id"]),
    ("Weekly: YearWeek + Part Name + Region Id", ["YearWeek", "Part Name", "Region Id"]),
    ("Monthly: YearMonth + Part Name + Region Id", ["YearMonth", "Part Name", "Region Id"]),
    ("Monthly: YearMonth + Bulk Value + Region Id", ["YearMonth", "Bulk Value", "Region Id"]),
]
grain_rows = []
for label, cols in key_candidates:
    unique_combos = len(df_raw.drop_duplicates(subset=cols))
    dup_rows = len(df_raw) - unique_combos
    grain_rows.append({
        "Grain / Candidate Key": label, "Total Raw Rows": len(df_raw),
        "Unique Key Combinations": unique_combos,
        "Duplicate / Multi-Transaction Rows": dup_rows,
        "Duplicate %": round(dup_rows / len(df_raw) * 100, 2),
        "Is Strictly Unique?": dup_rows == 0,
    })
df_grain_uniqueness = pd.DataFrame(grain_rows)
print("\n--- 1.5 & 1.6 GRAIN & KEY UNIQUENESS ---")
display(df_grain_uniqueness)
print("\nDay-of-month distribution (top 10):")
display(df_raw["Date"].dt.day.value_counts().head(10).rename_axis("Day_of_Month").to_frame("Record_Count"))

# 1.7 Data Volume
df_vol_year = (
    df_raw.groupby("Year", dropna=False)
    .agg(
        Records=("Quantity", "size"), Active_Months=("YearMonth", "nunique"),
        Unique_Parts=("Part Name", "nunique"), Unique_Bulks=("Bulk Value", "nunique"),
        Total_Quantity=("Quantity", "sum")
    )
    .reset_index()
    .merge(unique_part_regions(df_raw, ["Year"]), on="Year", how="left")
)
df_vol_month = (
    df_raw.groupby(["Year", "YearMonth"], dropna=False)
    .agg(Records=("Quantity", "size"), Unique_Parts=("Part Name", "nunique"), Total_Quantity=("Quantity", "sum"))
    .reset_index()
)
print("\n--- 1.7 DATA VOLUME BY YEAR ---")
display(df_vol_year)
print("\n--- 1.7 DATA VOLUME BY YEAR-MONTH ---")
display(df_vol_month)

## Cell 3: Section 2 — Missing Value Assessment (Raw Data)

In [ ]:
# ==========================================
# 2. MISSING VALUE ASSESSMENT (RAW DATA)
# ==========================================
orig_cols = EXPECTED_COLS
total_cells = df_raw[orig_cols].size
total_missing_cells = int(df_raw[orig_cols].isna().sum().sum())
print(f"Overall missing: {total_missing_cells:,} of {total_cells:,} cells ({total_missing_cells / total_cells * 100:.4f}%)")

def classify_severity(pct):
    if pct == 0:   return "None (0%)"
    if pct < 1.0:  return "Low (<1%)"
    if pct < 5.0:  return "Medium (1%-5%)"
    if pct < 20.0: return "High (5%-20%)"
    return "Critical (>20%)"

rows = []
for col in orig_cols:
    miss = int(df_raw[col].isna().sum())
    pct = round(miss / len(df_raw) * 100, 4)
    rows.append({
        "Column": col, "Missing Records": miss, "Populated Records": len(df_raw) - miss,
        "Missing %": pct, "Severity": classify_severity(pct)
    })
df_missing_col = pd.DataFrame(rows).sort_values("Missing Records", ascending=False)
print("\n--- 2.1 COLUMN-WISE MISSING VALUES & SEVERITY ---")
display(df_missing_col)

null_ind = df_raw[orig_cols].isna().astype(int)
null_ind["Year"] = df_raw["Year"]
null_ind["YearMonth"] = df_raw["YearMonth"].astype(str)

df_miss_by_year_cnt = null_ind.groupby("Year")[orig_cols].sum()
df_miss_by_year_pct = (null_ind.groupby("Year")[orig_cols].mean() * 100).round(2)
print("\n--- 2.2 MISSING RECORD COUNTS BY YEAR ---")
display(df_miss_by_year_cnt)
print("\n--- 2.2 MISSING % BY YEAR ---")
display(df_miss_by_year_pct)

df_miss_by_ym_cnt = null_ind.groupby("YearMonth")[orig_cols].sum()
print("\n--- 2.2 MISSING COUNTS BY YEAR-MONTH (months with >= 1 missing) ---")
display(df_miss_by_ym_cnt[df_miss_by_ym_cnt.sum(axis=1) > 0])

cols_with_nulls = [c for c in orig_cols if df_raw[c].isna().sum() > 0]
if len(cols_with_nulls) > 1:
    ni = df_raw[cols_with_nulls].isna().astype(int)
    df_miss_corr = ni.corr().round(3)
    df_miss_cooccur = ni.T.dot(ni)
    print("\n--- 2.3 MISSINGNESS CORRELATION ---")
    display(df_miss_corr)
    print("\n--- 2.3 MISSINGNESS CO-OCCURRENCE (simultaneous nulls) ---")
    display(df_miss_cooccur)
else:
    df_miss_corr, df_miss_cooccur = pd.DataFrame(), pd.DataFrame()
    print("\nFewer than 2 columns contain nulls; no null co-occurrence to analyse.")

## Cell 4A: Section 3.1 — Individual Master Data Mapping Tables
Each relationship is evaluated and displayed **separately** in both directions (Forward & Reverse), showing exact `1-to-1` vs. `1-to-many` counts.

In [ ]:
# ==========================================
# 3.1 INDIVIDUAL MASTER DATA MAPPING TABLES
# ==========================================
def build_pair_mapping_table(df, left_cols, right_cols, left_label, right_label, pair_title):
    left_list = [left_cols] if isinstance(left_cols, str) else list(left_cols)
    right_list = [right_cols] if isinstance(right_cols, str) else list(right_cols)
    all_cols = left_list + right_list

    sub = df[all_cols].dropna().drop_duplicates()

    # Forward: Left -> Right
    l_to_r = sub.groupby(left_list).size()
    n_left = int(len(l_to_r))
    left_1to1 = int((l_to_r == 1).sum())
    left_1tom = int((l_to_r > 1).sum())
    max_r_per_l = int(l_to_r.max()) if n_left > 0 else 0

    # Reverse: Right -> Left
    r_to_l = sub.groupby(right_list).size()
    n_right = int(len(r_to_l))
    right_1to1 = int((r_to_l == 1).sum())
    right_1tom = int((r_to_l > 1).sum())
    max_l_per_r = int(r_to_l.max()) if n_right > 0 else 0

    tbl = pd.DataFrame([
        {
            "Mapping_Pair": pair_title,
            "Direction": f"{left_label}  -->  {right_label}",
            "Distinct_Source_Values": n_left,
            "Maps_To_1_Target (1-to-1)": left_1to1,
            "Maps_To_Multiple_Targets (1-to-Many)": left_1tom,
            "One_To_Many_Pct": round(left_1tom / n_left * 100, 2) if n_left else 0.0,
            "Max_Targets_Per_Source": max_r_per_l,
            "Verdict": "Strict 1-to-1" if left_1tom == 0 else f"1-to-Many ({left_1tom:,} split)",
        },
        {
            "Mapping_Pair": pair_title,
            "Direction": f"{right_label}  -->  {left_label}",
            "Distinct_Source_Values": n_right,
            "Maps_To_1_Target (1-to-1)": right_1to1,
            "Maps_To_Multiple_Targets (1-to-Many)": right_1tom,
            "One_To_Many_Pct": round(right_1tom / n_right * 100, 2) if n_right else 0.0,
            "Max_Targets_Per_Source": max_l_per_r,
            "Verdict": "Strict 1-to-1" if right_1tom == 0 else f"1-to-Many ({right_1tom:,} shared)",
        },
    ])
    return tbl

MAPPING_SPECS = [
    ("3_1a_Part_to_Desc", "3.1.1 Part Name <-> Part Description", "Part Name", "Part Description", "Part Name", "Part Description"),
    ("3_1b_SegCode_to_Desc", "3.1.2 Business Segment Value <-> Description", "Business Segment Value", "Business Segment Description", "Segment Value", "Segment Description"),
    ("3_1c_Part_to_Bulk", "3.1.3 Part Name <-> Bulk Value", "Part Name", "Bulk Value", "Part Name", "Bulk Value"),
    ("3_1d_Part_to_Site", "3.1.4 Part Name <-> Part Site", "Part Name", "Part Site", "Part Name", "Part Site"),
    ("3_1e_Part_to_Region", "3.1.5 Part Name <-> Region Id", "Part Name", "Region Id", "Part Name", "Region Id"),
    ("3_1f_Site_to_Region", "3.1.6 Part Site <-> Region Id", "Part Site", "Region Id", "Part Site", "Region Id"),
    ("3_1g_Part_to_Segment", "3.1.7 Part Name <-> Business Segment Description", "Part Name", "Business Segment Description", "Part Name", "Segment Description"),
    ("3_1h_Bulk_to_Segment", "3.1.8 Bulk Value <-> Business Segment Description", "Bulk Value", "Business Segment Description", "Bulk Value", "Segment Description"),
    ("3_1i_PartReg_to_Bulk", "3.1.9 (Part Name + Region Id) <-> Bulk Value [Forecast -> WMAPE Roll-up]", ["Part Name", "Region Id"], "Bulk Value", "Part Name + Region Id", "Bulk Value"),
]

individual_mapping_tables = {}
for sheet_key, title, l_col, r_col, l_lbl, r_lbl in MAPPING_SPECS:
    tbl = build_pair_mapping_table(df_raw, l_col, r_col, l_lbl, r_lbl, title)
    individual_mapping_tables[sheet_key] = tbl
    print(f"\n=== {title} ===")
    display(tbl.drop(columns=["Mapping_Pair"]))

df_mapping_summary = pd.concat(individual_mapping_tables.values(), ignore_index=True)

## Cell 4B: Section 3.2 — One-to-Many Mapping Drill-Downs
Isolated inspection tables for specific codes that map to multiple descriptions, sites, regions, or business segments.

In [ ]:
# ==========================================
# 3.2 ONE-TO-MANY MAPPING DRILL-DOWNS
# ==========================================
def one_to_many(df, a, b):
    g = (
        df[[a, b]].dropna().drop_duplicates()
        .groupby(a)[b]
        .agg(Distinct_Target_Count="nunique", Mapped_Values=lambda s: " | ".join(sorted(s)[:8]))
        .reset_index()
    )
    return g[g["Distinct_Target_Count"] > 1].sort_values("Distinct_Target_Count", ascending=False).reset_index(drop=True)

df_map_seg_anom = one_to_many(df_raw, "Business Segment Value", "Business Segment Description")
df_map_site_reg_anom = one_to_many(df_raw, "Part Site", "Region Id")
df_map_part_seg_anom = one_to_many(df_raw, "Part Name", "Business Segment Description")
df_map_part_bulk_anom = one_to_many(df_raw, "Part Name", "Bulk Value")

print(f"1) Segment Codes mapped to > 1 Description: {len(df_map_seg_anom):,}")
if len(df_map_seg_anom):
    display(df_map_seg_anom.head(15))

print(f"\n2) Part Sites mapped to > 1 Region Id: {len(df_map_site_reg_anom):,}")
if len(df_map_site_reg_anom):
    display(df_map_site_reg_anom.head(15))

print(f"\n3) Part Names mapped to > 1 Business Segment: {len(df_map_part_seg_anom):,}")
if len(df_map_part_seg_anom):
    display(df_map_part_seg_anom.head(15))

print(f"\n4) Part Names mapped to > 1 Bulk Value: {len(df_map_part_bulk_anom):,}")
if len(df_map_part_bulk_anom):
    display(df_map_part_bulk_anom.head(15))

## Cell 4C: Section 3.3 — Orphan Records & Missing Master Attributes
Breakdown of raw rows missing at least one master attribute so you see exact combinations affected by cleaning.

In [ ]:
# ==========================================
# 3.3 ORPHAN RECORDS & MISSING MASTER ATTRIBUTES
# ==========================================
master_cols = [
    "Part Name", "Part Description", "Part Site", "Bulk Value",
    "Business Segment Value", "Business Segment Description", "Region Id"
]
orphan_mask = df_raw[master_cols].isna().any(axis=1)
print(f"Total raw records with >= 1 missing master attribute: {int(orphan_mask.sum()):,} ({orphan_mask.mean() * 100:.2f}%)")

df_orphan_patterns = (
    df_raw.loc[orphan_mask, master_cols].isna()
    .value_counts()
    .rename("Record_Count")
    .reset_index()
    .assign(Share_Of_Orphans_Pct=lambda d: (d["Record_Count"] / d["Record_Count"].sum() * 100).round(2))
)
display(df_orphan_patterns)

## Cell 5A: Section 4.1 — Coverage, Discontinuation & Low-Record Counts (`1`, `2`, `< 3` & `2025` Launches)

In [ ]:
# ==========================================
# 4.1 COVERAGE, DISCONTINUATION & LOW-RECORD BREAKDOWN
# ==========================================
cutoff_12m = (df_raw["Date"].max() - pd.DateOffset(months=11)).to_period("M").to_timestamp()
print(f"Last-12-months window: {cutoff_12m:%Y-%m-%d} to {df_raw['Date'].max():%Y-%m-%d}")

entity_levels = {
    "Part Name": ["Part Name"],
    "Part Name + Part Site": ["Part Name", "Part Site"],
    "Part Name + Region Id": ["Part Name", "Region Id"],
    "Bulk Value + Region Id": ["Bulk Value", "Region Id"],
    "Bulk Value + Part Site": ["Bulk Value", "Part Site"],
}

cov_rows = []
for lvl, cols in entity_levels.items():
    grp = df_raw.dropna(subset=cols).groupby(cols).agg(
        total_records=("Quantity", "size"),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        total_qty=("Quantity", "sum"),
    )
    no_last_12m = grp["last_date"] < cutoff_12m
    eq1 = grp["total_records"] == 1
    eq2 = grp["total_records"] == 2
    lt3 = grp["total_records"] < 3
    cov_rows.append({
        "Entity Level": lvl,
        "Total Entities": len(grp),
        "No Record in Last 12M (Discontinued)": int(no_last_12m.sum()),
        "No Record in Last 12M %": round(no_last_12m.mean() * 100, 2),
        "1 Record Only": int(eq1.sum()),
        "2 Records Only": int(eq2.sum()),
        "< 3 Records Total": int(lt3.sum()),
        "< 3 Records %": round(lt3.mean() * 100, 2),
        "Active in Last 12M & >= 3 Records": int((~no_last_12m & ~lt3).sum()),
    })
df_coverage_summary = pd.DataFrame(cov_rows)
print("\n--- 4.1 COVERAGE, DISCONTINUATION & LOW-RECORD SUMMARY (ALL YEARS) ---")
display(df_coverage_summary)

# 4.1b Introduced in 2025: 1 Record, 2 Records, < 3 Records breakdown (Part+Region & Bulk+Region)
intro_2025_rows = []
for lvl, cols in [("Part Name + Region Id", ["Part Name", "Region Id"]), ("Bulk Value + Region Id", ["Bulk Value", "Region Id"])]:
    sub = df_raw.dropna(subset=cols).copy()
    grp_all = sub.groupby(cols).agg(
        total_records=("Quantity", "size"),
        first_date=("Date", "min"),
        total_qty=("Quantity", "sum"),
    )
    # Basis A: First record appeared in 2025
    intro_rec = grp_all[grp_all["first_date"].dt.year == 2025]
    n_rec = len(intro_rec)
    intro_2025_rows.append({
        "Entity Level": lvl,
        "2025 Intro Basis": "First Record in 2025",
        "Total Introduced in 2025": n_rec,
        "1 Record": int((intro_rec["total_records"] == 1).sum()),
        "2 Records": int((intro_rec["total_records"] == 2).sum()),
        "< 3 Records (1 or 2)": int((intro_rec["total_records"] < 3).sum()),
        "< 3 Records %": round((intro_rec["total_records"] < 3).mean() * 100, 2) if n_rec else 0.0,
        ">= 3 Records": int((intro_rec["total_records"] >= 3).sum()),
        "Total Quantity": round(intro_rec["total_qty"].sum(), 2),
    })

    # Basis B: First POSITIVE demand (Quantity > 0) appeared in 2025
    pos_first = sub[sub["Quantity"] > 0].groupby(cols)["Date"].min().rename("first_pos_date")
    grp_pos = grp_all.join(pos_first, how="inner")
    intro_pos = grp_pos[grp_pos["first_pos_date"].dt.year == 2025]
    n_pos = len(intro_pos)
    intro_2025_rows.append({
        "Entity Level": lvl,
        "2025 Intro Basis": "First Positive Demand (>0) in 2025",
        "Total Introduced in 2025": n_pos,
        "1 Record": int((intro_pos["total_records"] == 1).sum()),
        "2 Records": int((intro_pos["total_records"] == 2).sum()),
        "< 3 Records (1 or 2)": int((intro_pos["total_records"] < 3).sum()),
        "< 3 Records %": round((intro_pos["total_records"] < 3).mean() * 100, 2) if n_pos else 0.0,
        ">= 3 Records": int((intro_pos["total_records"] >= 3).sum()),
        "Total Quantity": round(intro_pos["total_qty"].sum(), 2),
    })

df_intro_2025 = pd.DataFrame(intro_2025_rows)
print("\n--- 4.1b INTRODUCED IN 2025: 1 RECORD, 2 RECORDS & < 3 RECORDS BREAKDOWN ---")
display(df_intro_2025)

# 4.1c Discontinued breakdown by Region / Site / Bulk / Segment
def discontinued_by(dim):
    g = (
        df_raw.dropna(subset=["Part Name", dim])
        .groupby(["Part Name", dim], as_index=False)
        .agg(last_date=("Date", "max"))
    )
    g["Discontinued_12M"] = g["last_date"] < cutoff_12m
    out = (
        g.groupby(dim)
        .agg(Part_Entities=("Part Name", "count"), Discontinued=("Discontinued_12M", "sum"))
        .reset_index().rename(columns={dim: "Value"})
    )
    out["Active"] = out["Part_Entities"] - out["Discontinued"]
    out["Discontinued_Pct"] = (out["Discontinued"] / out["Part_Entities"] * 100).round(2)
    out.insert(0, "Dimension", dim)
    return out

df_discontinued = pd.concat(
    [discontinued_by(d) for d in ["Region Id", "Part Site", "Bulk Value", "Business Segment Description"]],
    ignore_index=True
)
print("\n--- 4.1c DISCONTINUED PART BREAKDOWN (by Region / Site / Bulk / Segment; top 12 per dimension) ---")
for d, sub in df_discontinued.groupby("Dimension", sort=False):
    print(f"\n[{d}]")
    display(sub.sort_values("Discontinued", ascending=False).head(12).reset_index(drop=True))

# 4.1d Restored: Discontinued (Part Name + Region Id) by Region x Business Segment
pr_grp = (
    df_raw.dropna(subset=["Part Name", "Region Id"])
    .groupby(["Part Name", "Region Id"], as_index=False)
    .agg(last_date=("Date", "max"), Business_Segment=("Business Segment Description", "first"))
)
pr_grp["Discontinued_12M"] = pr_grp["last_date"] < cutoff_12m
df_disc_reg_seg = (
    pr_grp.groupby(["Region Id", "Business_Segment"], dropna=False)
    .agg(Total_Part_Regions=("Part Name", "count"), Discontinued_Count=("Discontinued_12M", "sum"))
    .assign(Active_Count=lambda d: d["Total_Part_Regions"] - d["Discontinued_Count"])
    .reset_index()
)
print("\n--- 4.1d DISCONTINUED (PART NAME + REGION ID) BY REGION x SEGMENT (segment = first seen; see Cell 8 for full attribution) ---")
display(df_disc_reg_seg)

## Cell 5B: Section 4.2 & 4.3 — Reconciliations (Year-Wise Quarter, Part × Year, Segment × Year, Segment × Quarter) & Outliers

In [ ]:
# ==========================================
# 4.2 RECONCILIATION & CROSS-RECONCILIATION
# ==========================================
grand_qty, grand_recs = df_raw["Quantity"].sum(), len(df_raw)
df_raw["Quarter_Label"] = "Q" + df_raw["Date"].dt.quarter.astype(str)
df_raw["YearQuarter"] = df_raw["Date"].dt.to_period("Q").astype(str)

recon_tables = {}
for key, dim in {
    "Region": "Region Id",
    "Segment": "Business Segment Description",
    "Site": "Part Site",
    "Bulk": "Bulk Value",
    "Year": "Year",
    "Month": "YearMonth",
}.items():
    is_time = key in ("Year", "Month")
    rec = (
        df_raw.groupby(dim, dropna=False)
        .agg(Records=("Quantity", "size"), Total_Quantity=("Quantity", "sum"))
        .assign(
            Record_Share_Pct=lambda d: (d["Records"] / grand_recs * 100).round(2),
            Quantity_Share_Pct=lambda d: (d["Total_Quantity"] / grand_qty * 100).round(2),
        )
        .reset_index()
    )
    rec = rec.sort_values(dim if is_time else "Total_Quantity", ascending=is_time).reset_index(drop=True)
    recon_tables[key] = rec
    print(f"\n--- 4.2 RECONCILIATION BY {dim.upper()} ({len(rec):,} rows{'' if len(rec) <= 30 else ', top 30 shown'}) ---")
    display(rec.head(30))

# Year-Wise Quarter Reconciliation (Table + Year x Quarter Pivot)
recon_qtr_yearwise = (
    df_raw.groupby(["Year", "Quarter_Label"], dropna=False)
    .agg(Records=("Quantity", "size"), Total_Quantity=("Quantity", "sum"))
    .reset_index()
    .sort_values(["Year", "Quarter_Label"])
    .reset_index(drop=True)
)
yr_totals = recon_qtr_yearwise.groupby("Year")["Total_Quantity"].transform("sum")
recon_qtr_yearwise["Within_Year_Qty_Share_Pct"] = (recon_qtr_yearwise["Total_Quantity"] / yr_totals * 100).round(2)
recon_qtr_yearwise["Overall_Qty_Share_Pct"] = (recon_qtr_yearwise["Total_Quantity"] / grand_qty * 100).round(2)
recon_tables["Quarter_YearWise"] = recon_qtr_yearwise

cross_yr_qtr = pd.pivot_table(
    df_raw, values="Quantity", index="Year", columns="Quarter_Label",
    aggfunc="sum", fill_value=0, margins=True
)
print("\n--- 4.2 RECONCILIATION BY QUARTER (YEAR-WISE) ---")
display(recon_qtr_yearwise)
print("\n--- 4.2 YEAR x QUARTER QUANTITY MATRIX (RAW) ---")
display(cross_yr_qtr)

# Cross Reconciliation 1: Region Id x Year (Quantity)
print("\n--- 4.2 CROSS RECONCILIATION: REGION ID x YEAR (QUANTITY) ---")
cross_reg_yr = pd.pivot_table(
    df_raw, values="Quantity", index="Region Id", columns="Year",
    aggfunc="sum", fill_value=0, margins=True
)
display(cross_reg_yr)

# Cross Reconciliation 2: Business Segment Description x Year (Quantity)
print("\n--- 4.2 CROSS RECONCILIATION: BUSINESS SEGMENT x YEAR (QUANTITY) ---")
cross_seg_yr = pd.pivot_table(
    df_raw, values="Quantity", index="Business Segment Description", columns="Year",
    aggfunc="sum", fill_value=0, margins=True
)
display(cross_seg_yr)

# Cross Reconciliation 3 (Restored): Business Segment Description x Quarter (Quantity)
print("\n--- 4.2 CROSS RECONCILIATION: BUSINESS SEGMENT x QUARTER (QUANTITY) ---")
cross_seg_qtr = pd.pivot_table(
    df_raw, values="Quantity", index="Business Segment Description", columns="YearQuarter",
    aggfunc="sum", fill_value=0, margins=True
)
display(cross_seg_qtr)

# Cross Reconciliation 4: Unique Part Name x Year (Quantity)
cross_part_yr = pd.pivot_table(
    df_raw, values="Quantity", index="Part Name", columns="Year",
    aggfunc="sum", fill_value=0, margins=True
)
if "All" in cross_part_yr.index:
    part_body = cross_part_yr.drop(index="All").sort_values("All", ascending=False)
    cross_part_yr = pd.concat([part_body, cross_part_yr.loc[["All"]]])
print(f"\n--- 4.2 CROSS RECONCILIATION: UNIQUE PART NAME x YEAR (QUANTITY) [Top 30 of {len(cross_part_yr)-1:,} parts] ---")
display(cross_part_yr.head(30))

# ==========================================
# 4.3 OUTLIERS - TRANSACTION LEVEL (RAW ROWS)
# ==========================================
pos_qty = df_raw.loc[df_raw["Quantity"] > 0, "Quantity"]
q1, q3 = pos_qty.quantile(0.25), pos_qty.quantile(0.75)
iqr = q3 - q1
fence_15, fence_30 = q3 + 1.5 * iqr, q3 + 3.0 * iqr
p99, p999 = pos_qty.quantile(0.99), pos_qty.quantile(0.999)

def flag_iqr(df, keys, min_obs):
    g = df.loc[df["Quantity"] > 0].groupby(keys, dropna=False)["Quantity"]
    f = pd.concat([g.size().rename("n"), g.quantile(0.25).rename("q1"), g.quantile(0.75).rename("q3")], axis=1).reset_index()
    f["fence"] = np.where(f["n"] >= min_obs, f["q3"] + 1.5 * (f["q3"] - f["q1"]), np.nan)
    m = df[keys + ["Quantity"]].merge(f[keys + ["fence"]], on=keys, how="left")
    return m["Quantity"].values > m["fence"].values

SER_PR = ["Part Name", "Region Id"]
SER_PSRS = ["Part Name", "Part Site", "Region Id", "Business Segment Description"]
df_raw["Is_Outlier_PartRegion"] = flag_iqr(df_raw, SER_PR, MIN_OBS_SERIES_OUTLIER)
df_raw["Is_Outlier_PartSiteRegionSeg"] = flag_iqr(df_raw, SER_PSRS, MIN_OBS_SERIES_OUTLIER)

def ov(metric, mask):
    return {"Metric": metric, "Count": int(mask.sum()), "% of Total": round(mask.mean() * 100, 3)}

outlier_overview = pd.DataFrame([
    ov("Negative Quantity (< 0)", df_raw["Quantity"] < 0),
    ov("Zero Quantity (== 0)", df_raw["Quantity"] == 0),
    ov(f"GLOBAL IQR outliers (> {fence_15:,.2f}) - coarse screen only", df_raw["Quantity"] > fence_15),
    ov(f"GLOBAL extreme IQR outliers (> {fence_30:,.2f})", df_raw["Quantity"] > fence_30),
    ov(f"Quantity > P99 ({p99:,.2f})", df_raw["Quantity"] > p99),
    ov(f"Quantity > P99.9 ({p999:,.2f})", df_raw["Quantity"] > p999),
    ov(f"PER-SERIES IQR, Part+Region (>= {MIN_OBS_SERIES_OUTLIER} positive rows)", df_raw["Is_Outlier_PartRegion"]),
    ov(f"PER-SERIES IQR, Part+Site+Region+Segment (>= {MIN_OBS_SERIES_OUTLIER} positive rows)", df_raw["Is_Outlier_PartSiteRegionSeg"]),
])
print("\n--- 4.3 TRANSACTION-LEVEL OUTLIER OVERVIEW ---")
display(outlier_overview)

def hotspots(keys, flag):
    return (
        df_raw.groupby(keys, dropna=False)
        .agg(
            Total_Records=("Quantity", "size"), Outlier_Count=(flag, "sum"),
            Max_Quantity=("Quantity", "max"), Median_Quantity=("Quantity", "median")
        )
        .assign(Outlier_Frequency_Pct=lambda d: (d["Outlier_Count"] / d["Total_Records"] * 100).round(2))
        .query("Outlier_Count > 0")
        .sort_values(["Outlier_Count", "Max_Quantity"], ascending=False).reset_index()
    )

outlier_hotspots = hotspots(SER_PSRS, "Is_Outlier_PartSiteRegionSeg")
outlier_hotspots_pr = hotspots(SER_PR, "Is_Outlier_PartRegion")
print("\n--- 4.3 TOP OUTLIER HOTSPOTS: Part-Site-Region-Segment ---")
display(outlier_hotspots.head(20))

## Cell 6: Section 5 — Data Cleaning & Waterfall Audit
Applies your exact 4 rules (`Quantity < 0 -> 0`, drop blank `Business Segment Description`, drop `Part Name` starting with `'9'`, drop blank `Bulk Value`).

In [ ]:
# ==========================================
# 5. DATA CLEANING & WATERFALL AUDIT
# ==========================================
df_clean = df_raw.copy()

def snapshot(df, step, removed, modified, notes):
    return {
        "Step": step, "Rows Remaining": len(df), "Rows Removed": removed, "Rows Modified": modified,
        "% of Raw Rows Removed": round(removed / len(df_raw) * 100, 3),
        "Total Quantity": round(df["Quantity"].sum(), 2),
        "Distinct Part Names": int(df["Part Name"].nunique(dropna=True)),
        "Distinct (Part Name + Region Id)": len(df[["Part Name", "Region Id"]].dropna().drop_duplicates()),
        "Distinct (Bulk Value + Region Id)": len(df[["Bulk Value", "Region Id"]].dropna().drop_duplicates()),
        "Notes": notes,
    }

waterfall = [snapshot(df_clean, "0. Raw input", 0, 0, "After trimming, null-sentinel handling, date/qty parsing")]

neg_rows = df_clean[df_clean["Quantity"] < 0]
print(f"Negative quantity audit: {len(neg_rows):,} rows | sum = {neg_rows['Quantity'].sum():,.2f}")
if len(neg_rows):
    display(neg_rows.groupby(["Region Id", "Business Segment Description"], dropna=False)["Quantity"].agg(["count", "sum", "min"]))

# Step 1: Convert negative quantity to zero
neg_count = int((df_clean["Quantity"] < 0).sum())
df_clean["Quantity"] = df_clean["Quantity"].clip(lower=0)
waterfall.append(snapshot(df_clean, "1. Negative Quantity -> 0", 0, neg_count, f"Clipped {neg_count:,} negative rows to 0"))

# Step 2: Remove rows with blank Business Segment Description
before = len(df_clean)
df_clean = df_clean[df_clean["Business Segment Description"].notna()].copy()
waterfall.append(snapshot(df_clean, "2. Remove blank Business Segment Description", before - len(df_clean), 0, "Dropped blank segment description"))

# Step 3: Remove rows whose Part Name starts with '9'
before = len(df_clean)
df_clean = df_clean[~df_clean["Part Name"].fillna("").str.startswith("9")].copy()
waterfall.append(snapshot(df_clean, "3. Remove Part Name starting with '9'", before - len(df_clean), 0, "Dropped '9*' parts"))

# Step 4: Remove rows whose Bulk Value is blank
before = len(df_clean)
df_clean = df_clean[df_clean["Bulk Value"].notna()].copy()
waterfall.append(snapshot(df_clean, "4. Remove blank Bulk Value", before - len(df_clean), 0, "Dropped blank bulk value"))

# Step 5 (technical guardrail): drop rows missing Date, Quantity, Part Name, or Region Id
before = len(df_clean)
bad = df_clean[["Date", "Quantity", "Part Name", "Region Id"]].isna().any(axis=1)
df_clean = df_clean[~bad].copy()
waterfall.append(snapshot(df_clean, "5. Guardrail: null Date/Quantity/Part Name/Region Id", before - len(df_clean), 0, "Technical filter; expected ~0"))

df_waterfall = pd.DataFrame(waterfall)
print("\n--- 5.1 CLEANING WATERFALL ---")
display(df_waterfall)

df_overlap = (
    pd.DataFrame({
        "Blank_Segment_Desc": df_raw["Business Segment Description"].isna(),
        "Part_Starts_With_9": df_raw["Part Name"].fillna("").str.startswith("9"),
        "Blank_Bulk_Value": df_raw["Bulk Value"].isna(),
    })
    .value_counts().rename("Raw_Record_Count").reset_index()
)
print("\n--- 5.2 FILTER OVERLAP (ON RAW DATA) ---")
display(df_overlap)

## Cell 7: Section 6 — Post-Cleaning Core EDA (Tables, Quarterly YoY Comparison & Bar/Pie Charts)

In [ ]:
# ==========================================
# 6. POST-CLEANING CORE EDA
# ==========================================
print(f"Clean dataset: {df_clean.shape[0]:,} rows | {df_clean['Date'].min():%Y-%m-%d} to {df_clean['Date'].max():%Y-%m-%d}")
clean_qty_total = df_clean["Quantity"].sum()
clean_parts_total = df_clean["Part Name"].nunique()

eda_region = (
    df_clean.groupby("Region Id")
    .agg(
        Total_Records=("Quantity", "size"), Distinct_Part_Names=("Part Name", "nunique"),
        Distinct_Bulk_Values=("Bulk Value", "nunique"), Distinct_Part_Sites=("Part Site", "nunique"),
        Distinct_Segments=("Business Segment Description", "nunique"), Total_Quantity=("Quantity", "sum"),
        Mean_Tx_Quantity=("Quantity", "mean"), Median_Tx_Quantity=("Quantity", "median")
    )
    .assign(
        Part_Share_Pct=lambda d: (d["Distinct_Part_Names"] / clean_parts_total * 100).round(2),
        Quantity_Share_Pct=lambda d: (d["Total_Quantity"] / clean_qty_total * 100).round(2),
    )
    .sort_values("Total_Quantity", ascending=False).reset_index()
)
print("\n--- 6.1 EDA BY REGION ID ---")
display(eda_region)

seg_keys = ["Business Segment Value", "Business Segment Description"]
eda_segment = (
    df_clean.groupby(seg_keys)
    .agg(Total_Records=("Quantity", "size"), Distinct_Part_Names=("Part Name", "nunique"), Total_Quantity=("Quantity", "sum"))
    .reset_index()
    .merge(unique_part_regions(df_clean, seg_keys).rename(columns={"Unique_Part_Regions": "Distinct_Part_Regions"}), on=seg_keys, how="left")
    .assign(Quantity_Share_Pct=lambda d: (d["Total_Quantity"] / clean_qty_total * 100).round(2))
    .sort_values("Total_Quantity", ascending=False).reset_index(drop=True)
)
print("\n--- 6.2 EDA BY BUSINESS SEGMENT ---")
display(eda_segment)

ct_parts_reg_seg = pd.pivot_table(
    df_clean, values="Part Name", index="Business Segment Description",
    columns="Region Id", aggfunc="nunique", fill_value=0, margins=True, margins_name="Total"
)
print("\n--- 6.3 DISTINCT PART NAMES: SEGMENT x REGION ---")
display(ct_parts_reg_seg)

ct_qty_reg_seg = pd.pivot_table(
    df_clean, values="Quantity", index="Business Segment Description",
    columns="Region Id", aggfunc="sum", fill_value=0, margins=True, margins_name="Total"
)
print("\n--- 6.4 TOTAL QUANTITY: SEGMENT x REGION ---")
display(ct_qty_reg_seg)

# 6.5a Monthly Seasonality Matrix
df_clean["Month_Num"] = df_clean["Date"].dt.month
df_clean["Month_Name"] = df_clean["Date"].dt.strftime("%b")
seasonality_qty = pd.pivot_table(
    df_clean, values="Quantity", index=["Month_Num", "Month_Name"],
    columns="Year", aggfunc="sum", fill_value=0
)
print("\n--- 6.5a MONTHLY QUANTITY SEASONALITY BY YEAR (2022 is partial: Oct-Dec) ---")
display(seasonality_qty)

# 6.5b Quarterly Quantity Comparison Across Every Year (Post-Cleaning, with YoY % Growth)
df_clean["Quarter_Label"] = "Q" + df_clean["Date"].dt.quarter.astype(str)
qtr_piv = (
    pd.pivot_table(
        df_clean, values="Quantity", index="Quarter_Label", columns="Year",
        aggfunc="sum", fill_value=0
    )
    .reindex(index=["Q1", "Q2", "Q3", "Q4"], columns=YEARS, fill_value=0)
)
qtr_yoy_comp = qtr_piv.copy()
qtr_yoy_comp["Total_All_Years"] = qtr_yoy_comp[YEARS].sum(axis=1)
if 2023 in qtr_yoy_comp.columns and 2024 in qtr_yoy_comp.columns:
    qtr_yoy_comp["YoY_2024_vs_2023_Pct"] = np.where(
        qtr_yoy_comp[2023] > 0,
        ((qtr_yoy_comp[2024] - qtr_yoy_comp[2023]) / qtr_yoy_comp[2023] * 100).round(2),
        np.nan,
    )
if 2024 in qtr_yoy_comp.columns and 2025 in qtr_yoy_comp.columns:
    qtr_yoy_comp["YoY_2025_vs_2024_Pct"] = np.where(
        qtr_yoy_comp[2024] > 0,
        ((qtr_yoy_comp[2025] - qtr_yoy_comp[2024]) / qtr_yoy_comp[2024] * 100).round(2),
        np.nan,
    )

tot_row = {y: qtr_piv[y].sum() for y in YEARS}
tot_row["Total_All_Years"] = sum(tot_row[y] for y in YEARS)
if 2023 in tot_row and 2024 in tot_row:
    tot_row["YoY_2024_vs_2023_Pct"] = round((tot_row[2024] - tot_row[2023]) / tot_row[2023] * 100, 2) if tot_row[2023] > 0 else np.nan
if 2024 in tot_row and 2025 in tot_row:
    tot_row["YoY_2025_vs_2024_Pct"] = round((tot_row[2025] - tot_row[2024]) / tot_row[2024] * 100, 2) if tot_row[2024] > 0 else np.nan
qtr_yoy_comp = pd.concat([qtr_yoy_comp, pd.DataFrame([tot_row], index=["Total"])])
qtr_yoy_comp.index.name = "Quarter"

print("\n--- 6.5b QUARTERLY QUANTITY COMPARISON BY YEAR (POST-CLEANING, WITH YoY % CHANGE) ---")
display(qtr_yoy_comp)

# 6.6 ABC / Pareto at Part Name + Region Id
pareto_df = (
    df_clean.groupby(["Part Name", "Region Id"], as_index=False)
    .agg(Total_Quantity=("Quantity", "sum"))
    .sort_values("Total_Quantity", ascending=False).reset_index(drop=True)
)
tot = pareto_df["Total_Quantity"].sum()
cum = pareto_df["Total_Quantity"].cumsum()
pareto_df["Cum_Quantity_Pct"] = cum / tot * 100
prior_pct = (cum - pareto_df["Total_Quantity"]) / tot * 100
pareto_df["ABC_Class"] = np.select(
    [prior_pct < 80, prior_pct < 95],
    ["A (Top 80% Vol)", "B (Next 15% Vol)"], default="C (Bottom 5% Vol)"
)
abc_summary = (
    pareto_df.groupby("ABC_Class")
    .agg(Series_Count=("Part Name", "count"), Total_Quantity=("Total_Quantity", "sum")).reset_index()
    .assign(
        Series_Share_Pct=lambda d: (d["Series_Count"] / len(pareto_df) * 100).round(2),
        Volume_Share_Pct=lambda d: (d["Total_Quantity"] / tot * 100).round(2),
    )
)
print("\n--- 6.6 ABC / PARETO SUMMARY ---")
display(abc_summary)

# ==========================================
# 6.7 SECTION 6 BAR & PIE CHARTS
# ==========================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 4, figsize=(23, 11))

# 1. Bar: Total Quantity by Region Id
b1 = axes[0, 0].bar(eda_region["Region Id"], eda_region["Total_Quantity"], color="#2b5c8f")
axes[0, 0].set_title("Total Quantity by Region Id", fontsize=11, fontweight="bold")
axes[0, 0].set_ylabel("Total Quantity")
axes[0, 0].bar_label(b1, fmt="%.0f", padding=3, fontsize=7.5)

# 2. Bar: Distinct Part Names by Region Id
b2 = axes[0, 1].bar(eda_region["Region Id"], eda_region["Distinct_Part_Names"], color="#2e8b57")
axes[0, 1].set_title("Distinct Part Names by Region Id", fontsize=11, fontweight="bold")
axes[0, 1].set_ylabel("Distinct Part Names")
axes[0, 1].bar_label(b2, fmt="%d", padding=3, fontsize=7.5)

# 3. Pie: Quantity Share % by Region Id
axes[0, 2].pie(
    eda_region["Total_Quantity"], labels=eda_region["Region Id"],
    autopct="%1.1f%%", startangle=140, colors=sns.color_palette("Set2", len(eda_region)),
    wedgeprops={"edgecolor": "white", "linewidth": 1.2}
)
axes[0, 2].set_title("Volume Share (%) by Region Id", fontsize=11, fontweight="bold")

# 4. Grouped Bar: Quarterly Quantity Comparison Across Every Year (Q1-Q4 x 2022-2025)
qtr_labels = ["Q1", "Q2", "Q3", "Q4"]
x_q = np.arange(len(qtr_labels))
bar_w = 0.20
yr_colors = ["#8da0cb", "#1f77b4", "#2ca02c", "#ff7f0e"]
for idx_y, y in enumerate(YEARS):
    offset = (idx_y - (len(YEARS) - 1) / 2) * bar_w
    axes[0, 3].bar(x_q + offset, qtr_piv[y].values, width=bar_w, label=str(y), color=yr_colors[idx_y % len(yr_colors)])
axes[0, 3].set_xticks(x_q)
axes[0, 3].set_xticklabels(qtr_labels)
axes[0, 3].set_title("Quarterly Quantity Comparison by Year", fontsize=11, fontweight="bold")
axes[0, 3].set_ylabel("Total Quantity")
axes[0, 3].legend(title="Year", fontsize=8)

# 5. Horizontal Bar: Total Quantity by Business Segment
seg_plot = eda_segment.groupby("Business Segment Description", as_index=False)["Total_Quantity"].sum().sort_values("Total_Quantity")
b5 = axes[1, 0].barh(seg_plot["Business Segment Description"], seg_plot["Total_Quantity"], color="#d97724")
axes[1, 0].set_title("Total Quantity by Business Segment", fontsize=11, fontweight="bold")
axes[1, 0].set_xlabel("Total Quantity")
axes[1, 0].bar_label(b5, fmt="%.0f", padding=3, fontsize=7.5)

# 6. Pie: Volume Share % by Business Segment
seg_pie = seg_plot.sort_values("Total_Quantity", ascending=False)
axes[1, 1].pie(
    seg_pie["Total_Quantity"], labels=seg_pie["Business Segment Description"],
    autopct="%1.1f%%", startangle=140, colors=sns.color_palette("Paired", len(seg_pie)),
    textprops={"fontsize": 7.5}, wedgeprops={"edgecolor": "white", "linewidth": 1.2}
)
axes[1, 1].set_title("Volume Share (%) by Business Segment", fontsize=11, fontweight="bold")

# 7. Bar: ABC / Pareto Series Share % vs Volume Share %
x_abc = np.arange(len(abc_summary))
axes[1, 2].bar(x_abc - 0.18, abc_summary["Series_Share_Pct"], width=0.35, color="#4c72b0", label="% of Part+Region Series")
axes[1, 2].bar(x_abc + 0.18, abc_summary["Volume_Share_Pct"], width=0.35, color="#dd8452", label="% of Total Volume")
axes[1, 2].set_xticks(x_abc)
axes[1, 2].set_xticklabels(abc_summary["ABC_Class"], rotation=15, ha="right", fontsize=8)
axes[1, 2].set_title("ABC / Pareto: Series Share vs. Volume Share", fontsize=11, fontweight="bold")
axes[1, 2].set_ylabel("Percentage (%)")
axes[1, 2].legend(fontsize=8)

# 8. Pie: ABC / Pareto Volume Concentration
axes[1, 3].pie(
    abc_summary["Total_Quantity"], labels=abc_summary["ABC_Class"],
    autopct="%1.1f%%", startangle=140, colors=["#2b5c8f", "#6baed6", "#bdd7e7"],
    textprops={"fontsize": 8}, wedgeprops={"edgecolor": "white", "linewidth": 1.2}
)
axes[1, 3].set_title("ABC / Pareto Volume Share (%)", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_SEC6_PNG, dpi=200)
plt.show()

## Cell 8: Section 7 — `Part Name + Region Id` Coverage, Year Iterations, Quarterly Cohorts & `Bulk + Region` Check

In [ ]:
# ==========================================
# 7. PART NAME + REGION ID: COVERAGE, LIFECYCLE & YEAR ITERATIONS
# ==========================================
KEY = ["Part Name", "Region Id"]

monthly_agg = (
    df_clean.groupby(KEY + ["YearMonth"], as_index=False)
    .agg(Monthly_Quantity=("Quantity", "sum"), Tx_Count=("Quantity", "size"))
)
monthly_agg["Year"] = monthly_agg["YearMonth"].dt.year
monthly_agg["Cal_Month"] = monthly_agg["YearMonth"].dt.month
monthly_agg["MonthIdx"] = monthly_agg["Year"] * 12 + monthly_agg["Cal_Month"]
monthly_agg["Is_Pos"] = monthly_agg["Monthly_Quantity"] > 0

full_ym_index = pd.period_range(df_clean["YearMonth"].min(), df_clean["YearMonth"].max(), freq="M")
TOTAL_CAL_MONTHS = len(full_ym_index)
WIN_START_IDX = full_ym_index[0].year * 12 + full_ym_index[0].month
WIN_END_IDX = full_ym_index[-1].year * 12 + full_ym_index[-1].month
print(f"Modeling window: {full_ym_index[0]} to {full_ym_index[-1]} ({TOTAL_CAL_MONTHS} months)")

def idx_to_period_str(s):
    out = pd.Series(np.nan, index=s.index, dtype="object")
    v = s.dropna()
    out.loc[v.index] = ((v - 1) // 12).astype(int).astype(str) + "-" + ((v - 1) % 12 + 1).astype(int).astype(str).str.zfill(2)
    return out

# 7.0 Monthly-grain outliers (per-series IQR cross-checked against same calendar month in other years)
mp = monthly_agg[monthly_agg["Is_Pos"]].copy()
_g = mp.groupby(KEY)["Monthly_Quantity"]
mfence = pd.concat([_g.size().rename("n"), _g.quantile(0.25).rename("q1"), _g.quantile(0.75).rename("q3")], axis=1)
_iqr = mfence["q3"] - mfence["q1"]
_ok = mfence["n"] >= MIN_OBS_MONTHLY_OUTLIER
mfence["hi"] = np.where(_ok, mfence["q3"] + 1.5 * _iqr, np.nan)
mfence["lo"] = np.where(_ok, np.maximum(mfence["q1"] - 1.5 * _iqr, 0), np.nan)
_look = mfence.reindex(pd.MultiIndex.from_frame(mp[KEY]))
mp["Fence_Hi"], mp["Fence_Lo"] = _look["hi"].values, _look["lo"].values
_gm = mp.groupby(KEY + ["Cal_Month"])["Monthly_Quantity"]
_sum, _cnt = _gm.transform("sum"), _gm.transform("count")
mp["Same_Month_Other_Years_Mean"] = (_sum - mp["Monthly_Quantity"]) / (_cnt - 1).where(_cnt > 1)
q, ref = mp["Monthly_Quantity"], mp["Same_Month_Other_Years_Mean"]
hi, lo = q > mp["Fence_Hi"], q < mp["Fence_Lo"]
mp["Monthly_Outlier_Type"] = np.select(
    [hi & ref.isna(), hi & (q <= SEASONAL_TOL * ref), hi,
     lo & ref.isna(), lo & (q >= ref / SEASONAL_TOL), lo],
    [
        "High - no seasonal reference",
        "High - in line with same month other years (likely seasonal)",
        "High - unexplained spike",
        "Low - no seasonal reference",
        "Low - in line with same month other years (likely seasonal)",
        "Low - unexplained dip",
    ],
    default=""
)
monthly_agg["Monthly_Outlier_Type"] = ""
monthly_agg.loc[mp.index, "Monthly_Outlier_Type"] = mp["Monthly_Outlier_Type"]
_unexpl = mp[mp["Monthly_Outlier_Type"].isin(["High - unexplained spike", "Low - unexplained dip"])]
unexpl_by_series = _unexpl.groupby(KEY).size()

# Attribute collapsing: preserve ALL distinct values joined by " | " plus distinct counts
def collapse_attr(col, out_name, n_name):
    d = df_clean[KEY + [col]].dropna().drop_duplicates()
    g = d.groupby(KEY)[col]
    return pd.concat([g.agg(lambda s: " | ".join(sorted(s))[:30000]).rename(out_name), g.size().rename(n_name)], axis=1)

attrs = pd.concat([
    collapse_attr("Part Description", "Part_Description", "N_Part_Descriptions"),
    collapse_attr("Bulk Value", "Bulk_Value", "N_Bulk_Values"),
    collapse_attr("Business Segment Description", "Business_Segment", "N_Business_Segments"),
    collapse_attr("Part Site", "Part_Site", "N_Part_Sites"),
], axis=1)

# Vectorized series catalog
cat = monthly_agg.groupby(KEY).agg(
    First_Idx=("MonthIdx", "min"), Last_Idx=("MonthIdx", "max"),
    Recorded_Months=("MonthIdx", "nunique"), Total_Quantity=("Monthly_Quantity", "sum")
)
cat = cat.join(attrs, how="left")
for c in ["N_Part_Descriptions", "N_Bulk_Values", "N_Business_Segments", "N_Part_Sites"]:
    cat[c] = cat[c].fillna(0).astype(int)

pos = monthly_agg[monthly_agg["Is_Pos"]].assign(Sq=lambda d: d["Monthly_Quantity"] ** 2)
pos_agg = pos.groupby(KEY).agg(
    Positive_Demand_Months=("MonthIdx", "nunique"),
    First_Pos_Idx=("MonthIdx", "min"), Last_Pos_Idx=("MonthIdx", "max"),
    Sum_Pos=("Monthly_Quantity", "sum"), SumSq_Pos=("Sq", "sum")
)
cat = cat.join(pos_agg, how="left")
cat["Positive_Demand_Months"] = cat["Positive_Demand_Months"].fillna(0).astype(int)
cat["Monthly_Unexplained_Outliers"] = unexpl_by_series.reindex(cat.index).fillna(0).astype(int)

def by_year(df, values, agg):
    return (
        df.pivot_table(index=KEY, columns="Year", values=values, aggfunc=agg, fill_value=0)
        .reindex(index=cat.index, columns=YEARS, fill_value=0)
    )

rec_by_year = by_year(monthly_agg, "MonthIdx", "nunique")
pos_by_year = by_year(pos, "MonthIdx", "nunique") if len(pos) else rec_by_year * 0
qty_by_year = by_year(monthly_agg, "Monthly_Quantity", "sum")
for y in YEARS:
    cat[f"Months_{y}"] = rec_by_year[y].fillna(0).astype(int)
    cat[f"PosMonths_{y}"] = pos_by_year[y].fillna(0).astype(int)
    cat[f"Qty_{y}"] = qty_by_year[y].fillna(0).round(2)
    cat[f"Has_{y}"] = (cat[f"Months_{y}"] > 0).astype(int)
    cat[f"Pos_{y}"] = (cat[f"PosMonths_{y}"] > 0).astype(int)

cat["Calendar_Span_Months"] = cat["Last_Idx"] - cat["First_Idx"] + 1
cat["First_Month"] = idx_to_period_str(cat["First_Idx"])
cat["Last_Month"] = idx_to_period_str(cat["Last_Idx"])
cat["First_Pos_Month"] = idx_to_period_str(cat["First_Pos_Idx"])
cat["Last_Pos_Month"] = idx_to_period_str(cat["Last_Pos_Idx"])
cat["Zero_Record_Months"] = cat["Recorded_Months"] - cat["Positive_Demand_Months"]
cat["Months_Since_Last_Positive"] = WIN_END_IDX - cat["Last_Pos_Idx"]
cat["Months_Of_History"] = WIN_END_IDX - cat["First_Pos_Idx"] + 1
cat["Possibly_Pre_Window"] = (cat["First_Pos_Idx"] == WIN_START_IDX).astype(int)

# Launch cohort labels: by First Positive Year and First Positive Quarter (2022-Q4 .. 2025-Q4)
_fp = cat["First_Pos_Idx"]
_fy, _fm = (_fp - 1) // 12, (_fp - 1) % 12 + 1
_fq = (_fm - 1) // 3 + 1
cat["First_Pos_Year"] = np.where(_fp.notna(), _fy.astype("Int64").astype(str), "Never Positive")
cat["First_Pos_Quarter"] = np.where(
    _fp.notna(),
    _fy.astype("Int64").astype(str) + "-Q" + _fq.astype("Int64").astype(str),
    "Never Positive"
)

# Syntetos-Boylan demand pattern classification
n = cat["Positive_Demand_Months"]
n_safe = n.where(n > 0)
mean_pos = cat["Sum_Pos"] / n_safe
var_pos = (cat["SumSq_Pos"] / n_safe - mean_pos ** 2).clip(lower=0)
cat["Avg_Monthly_Qty_Active"] = mean_pos.fillna(0).round(2)
cat["ADI"] = ((WIN_END_IDX - cat["First_Pos_Idx"] + 1) / n_safe).round(2)
cat["CV2"] = (var_pos / mean_pos ** 2).where(n > 1, 0.0).round(2)
cat["Demand_Pattern"] = np.select(
    [n == 0, (cat["ADI"] < 1.32) & (cat["CV2"] < 0.49), cat["ADI"] < 1.32, cat["CV2"] < 0.49],
    ["Zero-Demand Only", "Smooth", "Erratic", "Intermittent"], default="Lumpy"
)
cat["Total_Quantity"] = cat["Total_Quantity"].round(2)

# Year-iteration labels
def iter_label(df, prefix, years, empty):
    parts = [np.where(df[f"{prefix}_{y}"] == 1, str(y), "") for y in years]
    lab = pd.Series(["+".join(p for p in tup if p) for tup in zip(*parts)], index=df.index)
    return lab.replace("", empty)

cat["Year_Iteration_2022_2025"] = iter_label(cat, "Has", YEARS, "None")
cat["Year_Iteration_2023_2025"] = iter_label(cat, "Has", YEARS[1:], f"{YEARS[0]} Only")
cat["Year_Iteration_Positive"] = iter_label(cat, "Pos", YEARS, "None")
cat["Year_Iteration_Pos_2023_2025"] = iter_label(cat, "Pos", YEARS[1:], f"None in {YEARS[1]}-{YEARS[-1]}")

# Forecasting tiers (positive-demand based)
L, L1, L2 = YEARS[-1], YEARS[-2], YEARS[-3]
cat["Forecasting_Tier"] = np.select(
    [
        n == 0,
        cat[f"Pos_{L}"] == 0,
        (cat[f"Pos_{L2}"] == 1) & (cat[f"Pos_{L1}"] == 1) & (cat[f"Pos_{L}"] == 1) & (n >= T1_MIN_POS_MONTHS),
        (cat[f"Pos_{L1}"] == 1) & (cat[f"Pos_{L}"] == 1) & (n >= T2_MIN_POS_MONTHS),
    ],
    [
        "Tier 5: Zero-Demand History (Exclude)",
        f"Tier 4: Dormant / No {L} Demand (Exclude or Zero-Forecast)",
        f"Tier 1: Full Multi-Year ({L2}-{L}, >={T1_MIN_POS_MONTHS} pos mos) - Seasonal/ML Ready",
        f"Tier 2: 2-Year ({L1}-{L}, >={T2_MIN_POS_MONTHS} pos mos) - Non-Seasonal/ETS/Croston",
    ],
    default=f"Tier 3: Short / New Launch / Sparse (<{T2_MIN_POS_MONTHS} pos mos or gaps) - Moving Avg / Croston / Analog"
)

active_L = cat[f"Pos_{L}"] == 1
cat["Lifecycle_Status"] = np.select(
    [n == 0, (_fy < L1) & active_L, (_fy < L1) & ~active_L, (_fy == L1) & active_L, (_fy == L1) & ~active_L, (_fy == L)],
    ["Zero-Demand Only", "Established Active", "Legacy Discontinued",
     f"{L1} Launch - Active", f"{L1} Launch - Already Churned", f"{L} Launch - New / Cold-Start"],
    default="Unclassified"
)

series_idx = cat[["First_Pos_Idx", "Last_Pos_Idx"]].reset_index()
series_catalog = cat.drop(columns=["First_Idx", "Last_Idx", "First_Pos_Idx", "Last_Pos_Idx", "Sum_Pos", "SumSq_Pos"]).reset_index()
total_series = len(series_catalog)
total_clean_qty = series_catalog["Total_Quantity"].sum()

def agg_iter(df, label_col):
    return (
        df.groupby(label_col)
        .agg(
            Part_Region_Count=("Part Name", "size"), Distinct_Parts=("Part Name", "nunique"),
            Total_Quantity=("Total_Quantity", "sum"),
            Avg_Recorded_Months=("Recorded_Months", "mean"), Median_Recorded_Months=("Recorded_Months", "median"),
            Avg_Positive_Months=("Positive_Demand_Months", "mean"), Median_Positive_Months=("Positive_Demand_Months", "median")
        )
        .assign(
            Series_Share_Pct=lambda d: (d["Part_Region_Count"] / total_series * 100).round(2),
            Quantity_Share_Pct=lambda d: (d["Total_Quantity"] / total_clean_qty * 100).round(2),
        )
        .reset_index()
    )

_zero_fill = {"Part_Region_Count": 0, "Distinct_Parts": 0, "Total_Quantity": 0, "Series_Share_Pct": 0, "Quantity_Share_Pct": 0}
combo_labels = ["+".join(map(str, c)) for r in range(1, len(YEARS) + 1) for c in itertools.combinations(YEARS, r)]

# 7.1 & 7.1b All 15 Year Combinations
report_all_15_iters = (
    pd.DataFrame({"Year_Iteration_2022_2025": combo_labels})
    .merge(agg_iter(series_catalog, "Year_Iteration_2022_2025"), how="left").fillna(_zero_fill)
    .sort_values("Part_Region_Count", ascending=False).reset_index(drop=True)
)
print(f"\n--- 7.1 ALL {len(combo_labels)} YEAR COMBINATIONS (ROW EXISTENCE BASIS) ---")
display(report_all_15_iters)

report_pos_iters = (
    pd.DataFrame({"Year_Iteration_Positive": combo_labels + ["None"]})
    .merge(agg_iter(series_catalog, "Year_Iteration_Positive"), how="left").fillna(_zero_fill)
    .sort_values("Part_Region_Count", ascending=False).reset_index(drop=True)
)
print(f"\n--- 7.1b ALL {len(combo_labels)} YEAR COMBINATIONS (POSITIVE DEMAND > 0 BASIS) ---")
display(report_pos_iters)

# 7.2 & 7.2b Collapsed 3-Year Iterations (Positive Demand & Restored Row-Presence Reference)
report_3yr_iters = agg_iter(series_catalog, "Year_Iteration_Pos_2023_2025").sort_values("Part_Region_Count", ascending=False).reset_index(drop=True)
report_3yr_rows = agg_iter(series_catalog, "Year_Iteration_2023_2025").sort_values("Part_Region_Count", ascending=False).reset_index(drop=True)
print(f"\n--- 7.2 COLLAPSED {YEARS[1]}-{YEARS[-1]} ITERATIONS (DEMAND > 0 BASIS) ---")
display(report_3yr_iters)
print(f"\n--- 7.2b COLLAPSED {YEARS[1]}-{YEARS[-1]} ITERATIONS (ROW-PRESENCE BASIS - REFERENCE) ---")
display(report_3yr_rows)

# 7.3 Regional Breakdown
ct_iter_region = pd.pivot_table(
    series_catalog, values="Part Name", index="Year_Iteration_Pos_2023_2025",
    columns="Region Id", aggfunc="count", fill_value=0, margins=True, margins_name="Total"
)
print("\n--- 7.3 YEAR ITERATION (DEMAND > 0) COUNT BY REGION ---")
display(ct_iter_region)

# 7.4 Forecasting Eligibility Tiers
tier_summary = (
    series_catalog.groupby("Forecasting_Tier")
    .agg(
        Part_Region_Count=("Part Name", "count"), Total_Quantity=("Total_Quantity", "sum"),
        Avg_Positive_Months=("Positive_Demand_Months", "mean")
    )
    .assign(
        Series_Share_Pct=lambda d: (d["Part_Region_Count"] / total_series * 100).round(2),
        Quantity_Share_Pct=lambda d: (d["Total_Quantity"] / total_clean_qty * 100).round(2),
    )
    .reset_index().sort_values("Forecasting_Tier")
)
print("\n--- 7.4 FORECASTING ELIGIBILITY TIERS (PART + REGION) ---")
display(tier_summary)

# 7.5 Demand Patterns (SBC)
active_last = series_catalog[series_catalog[f"Pos_{L}"] == 1]
sbc_summary = (
    active_last.groupby("Demand_Pattern")
    .agg(
        Part_Region_Count=("Part Name", "count"), Total_Quantity=("Total_Quantity", "sum"),
        Median_Positive_Months=("Positive_Demand_Months", "median")
    )
    .assign(
        Series_Share_Pct=lambda d: (d["Part_Region_Count"] / len(active_last) * 100).round(2),
        Quantity_Share_Pct=lambda d: (d["Total_Quantity"] / active_last["Total_Quantity"].sum() * 100).round(2),
    )
    .reset_index()
)
print(f"\n--- 7.5 DEMAND PATTERNS (SERIES WITH POSITIVE DEMAND IN {L}) ---")
display(sbc_summary)

# 7.6 Month Buckets
edges = [e for e in [0, 3, 6, 12, 18, 24, 30, 36] if e < TOTAL_CAL_MONTHS] + [TOTAL_CAL_MONTHS]
labels = [f"{a + 1}-{b} mos" for a, b in zip(edges[:-1], edges[1:])]
series_catalog["Recorded_Months_Bucket"] = pd.cut(series_catalog["Recorded_Months"], bins=edges, labels=labels)
series_catalog["Positive_Months_Bucket"] = pd.cut(series_catalog["Positive_Demand_Months"], bins=[-1] + edges, labels=["0 mos"] + labels)
print("\n--- 7.6 RECORDED vs. POSITIVE-DEMAND MONTHS PER SERIES ---")
display(pd.DataFrame({
    "Recorded_Months_Series": series_catalog["Recorded_Months_Bucket"].value_counts().sort_index(),
    "Positive_Months_Series": series_catalog["Positive_Months_Bucket"].value_counts().sort_index(),
}).fillna(0).astype(int))

# 7.7 Launch Cohorts by First-Positive Year and Quarter
total_last_qty = series_catalog[f"Qty_{L}"].sum()
def cohort_table(label_col, level):
    out = (
        series_catalog.groupby(label_col)
        .agg(
            Series_Count=("Part Name", "size"), Total_Quantity=("Total_Quantity", "sum"),
            Qty_Last_Year=(f"Qty_{L}", "sum"), Median_Positive_Months=("Positive_Demand_Months", "median"),
            Median_History_Months=("Months_Of_History", "median"),
            Possibly_Pre_Window=("Possibly_Pre_Window", "sum")
        )
        .reset_index().rename(columns={label_col: "Cohort"})
    )
    out["Series_Share_Pct"] = (out["Series_Count"] / total_series * 100).round(2)
    out["Quantity_Share_Pct"] = (out["Total_Quantity"] / total_clean_qty * 100).round(2)
    out[f"Share_of_{L}_Volume_Pct"] = (out["Qty_Last_Year"] / total_last_qty * 100).round(2) if total_last_qty else 0.0
    out = out.assign(_never=out["Cohort"] == "Never Positive").sort_values(["_never", "Cohort"]).drop(columns="_never")
    out.insert(0, "Cohort_Level", level)
    return out.reset_index(drop=True)

cohort_year = cohort_table("First_Pos_Year", "Year")
cohort_quarter = cohort_table("First_Pos_Quarter", "Quarter")
launch_cohorts = pd.concat([cohort_year, cohort_quarter], ignore_index=True)

print("\n--- 7.7a LAUNCH COHORTS BY FIRST-POSITIVE YEAR ---")
display(cohort_year)
print("\n--- 7.7b LAUNCH COHORTS BY FIRST-POSITIVE QUARTER ---")
display(cohort_quarter)

lifecycle_summary = (
    series_catalog.groupby("Lifecycle_Status")
    .agg(
        Series_Count=("Part Name", "size"), Total_Quantity=("Total_Quantity", "sum"),
        Qty_Last_Year=(f"Qty_{L}", "sum"), Median_Positive_Months=("Positive_Demand_Months", "median")
    )
    .assign(
        Series_Share_Pct=lambda d: (d["Series_Count"] / total_series * 100).round(2),
        Quantity_Share_Pct=lambda d: (d["Total_Quantity"] / total_clean_qty * 100).round(2),
    )
    .reset_index()
)
print("\n--- 7.7c LIFECYCLE STATUS SUMMARY ---")
display(lifecycle_summary)

# 7.8 Monthly-Grain Outliers
n_pos_months = len(mp)
monthly_outlier_summary = (
    mp.loc[mp["Monthly_Outlier_Type"] != "", "Monthly_Outlier_Type"].value_counts()
    .rename("Series_Months").reset_index().rename(columns={"index": "Monthly_Outlier_Type"})
)
monthly_outlier_summary["Pct_of_Positive_Series_Months"] = (monthly_outlier_summary["Series_Months"] / n_pos_months * 100).round(3)
print(f"\n--- 7.8 MONTHLY-GRAIN OUTLIERS ({n_pos_months:,} positive series-months evaluated) ---")
display(monthly_outlier_summary)

monthly_outlier_rows = (
    mp[mp["Monthly_Outlier_Type"].isin(["High - unexplained spike", "Low - unexplained dip"])]
    [KEY + ["YearMonth", "Monthly_Quantity", "Fence_Lo", "Fence_Hi", "Same_Month_Other_Years_Mean", "Monthly_Outlier_Type"]]
    .assign(Ratio_vs_Same_Month=lambda d: (d["Monthly_Quantity"] / d["Same_Month_Other_Years_Mean"]).round(2))
    .sort_values("Monthly_Quantity", ascending=False).reset_index(drop=True)
)
print("\nTop 20 unexplained monthly outliers (review, don't auto-clip):")
display(monthly_outlier_rows.head(20))

# 7.9 Lightweight Bulk Value + Region Id WMAPE Evaluation Coverage Check
bulk_reg_clean = (
    df_clean.groupby(["Bulk Value", "Region Id"], as_index=False)
    .agg(
        Part_Count=("Part Name", "nunique"),
        Total_Quantity=("Quantity", "sum"),
        Qty_Last_Year=("Quantity", lambda s: s[df_clean.loc[s.index, "Year"] == L].sum()),
    )
)
bulk_reg_clean["Active_In_Last_Year"] = bulk_reg_clean["Qty_Last_Year"] > 0
bulk_wmape_summary = pd.DataFrame([
    {"Metric": "Total Clean (Bulk Value + Region Id) Pairs", "Value": f"{len(bulk_reg_clean):,}"},
    {"Metric": f"Active in {L} (Positive Volume > 0)", "Value": f"{int(bulk_reg_clean['Active_In_Last_Year'].sum()):,} ({bulk_reg_clean['Active_In_Last_Year'].mean()*100:.1f}%)"},
    {"Metric": f"Dormant / Zero in {L}", "Value": f"{int((~bulk_reg_clean['Active_In_Last_Year']).sum()):,}"},
    {"Metric": "1-to-1 Bulk+Region (Single Part Name)", "Value": f"{int((bulk_reg_clean['Part_Count'] == 1).sum()):,} ({(bulk_reg_clean['Part_Count'] == 1).mean()*100:.1f}%)"},
    {"Metric": "Many-to-1 Bulk+Region (>= 2 Part Names)", "Value": f"{int((bulk_reg_clean['Part_Count'] > 1).sum()):,} ({(bulk_reg_clean['Part_Count'] > 1).mean()*100:.1f}%)"},
    {"Metric": "Max Part Names Rolling into One Bulk+Region", "Value": f"{int(bulk_reg_clean['Part_Count'].max()):,}"},
])
print("\n--- 7.9 BULK VALUE + REGION ID (WMAPE EVALUATION LEVEL) COVERAGE CHECK ---")
display(bulk_wmape_summary)

## Cell 9: Section 8 — Visual EDA & Year-Iteration Readiness Dashboard

In [ ]:
# ==========================================
# 8. VISUAL EDA & READINESS CHARTS
# ==========================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

monthly_total = df_clean.groupby("YearMonth")["Quantity"].sum().reset_index()
axes[0, 0].plot(monthly_total["YearMonth"].astype(str), monthly_total["Quantity"], marker="o", linewidth=2, color="#1f77b4")
axes[0, 0].set_title(f"Monthly Total Quantity ({full_ym_index[0]} to {full_ym_index[-1]})", fontsize=12, fontweight="bold")
axes[0, 0].tick_params(axis="x", rotation=75)
axes[0, 0].set_ylabel("Total Quantity")

ax2 = axes[0, 1]
ax2_twin = ax2.twinx()
x_pos = np.arange(len(eda_region))
ax2.bar(x_pos - 0.18, eda_region["Distinct_Part_Names"], width=0.35, color="#2ca02c")
ax2_twin.bar(x_pos + 0.18, eda_region["Total_Quantity"], width=0.35, color="#ff7f0e", alpha=0.8)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(eda_region["Region Id"])
ax2.set_title("Distinct Part Names (green) vs Volume (orange) by Region Id", fontsize=12, fontweight="bold")
ax2.set_ylabel("Distinct Part Names", color="#2ca02c")
ax2_twin.set_ylabel("Total Quantity", color="#ff7f0e")
ax2_twin.grid(False)

ax3 = axes[1, 0]
x_iter = np.arange(len(report_3yr_iters))
ax3.bar(x_iter - 0.18, report_3yr_iters["Series_Share_Pct"], width=0.35, color="#4c72b0", label="% of Part+Region series")
ax3.bar(x_iter + 0.18, report_3yr_iters["Quantity_Share_Pct"], width=0.35, color="#dd8452", label="% of total quantity")
ax3.set_xticks(x_iter)
ax3.set_xticklabels(report_3yr_iters["Year_Iteration_Pos_2023_2025"], rotation=35, ha="right")
ax3.set_title("Year Iterations (demand > 0): Series Share vs Volume Share", fontsize=12, fontweight="bold")
ax3.set_ylabel("Percentage (%)")
ax3.legend()

pos_counts = series_catalog["Positive_Months_Bucket"].value_counts().sort_index()
axes[1, 1].bar(pos_counts.index.astype(str), pos_counts.values, color="#55a868")
axes[1, 1].set_title("Positive-Demand Months per Part Name + Region Id", fontsize=12, fontweight="bold")
axes[1, 1].tick_params(axis="x", rotation=35)
axes[1, 1].set_ylabel("Number of series")

plt.tight_layout()
plt.savefig(OUTPUT_DASHBOARD_PNG, dpi=200)
plt.show()

## Cell 10: Section 9 — Export Excel Audit Report & Dense Monthly Panel CSV

In [ ]:
# ==========================================
# 9. EXPORT EXCEL REPORT & MODELING PANEL
# ==========================================
panel_attr_cols = [
    "Part_Description", "Bulk_Value", "Business_Segment", "First_Pos_Month", "First_Pos_Quarter",
    "Last_Pos_Month", "Lifecycle_Status", "Year_Iteration_Positive", "Forecasting_Tier"
]
unique_pr = series_catalog[KEY + panel_attr_cols].merge(series_idx, on=KEY, how="left")
print(f"Dense panel size: {len(unique_pr):,} series x {TOTAL_CAL_MONTHS} months = {len(unique_pr) * TOTAL_CAL_MONTHS:,} rows")

dense_grid = unique_pr.merge(pd.DataFrame({"YearMonth": full_ym_index}), how="cross")
dense = dense_grid.merge(
    monthly_agg[KEY + ["YearMonth", "Monthly_Quantity", "Tx_Count", "Monthly_Outlier_Type"]],
    on=KEY + ["YearMonth"], how="left"
)
_mi = dense["YearMonth"].dt.year * 12 + dense["YearMonth"].dt.month
dense["Is_Recorded_Month"] = dense["Monthly_Quantity"].notna().astype(int)
dense["Monthly_Quantity"] = dense["Monthly_Quantity"].fillna(0.0)
dense["Tx_Count"] = dense["Tx_Count"].fillna(0).astype(int)
dense["Monthly_Outlier_Type"] = dense["Monthly_Outlier_Type"].fillna("")
dense["Date"] = dense["YearMonth"].dt.to_timestamp()
dense["Is_Post_First_Positive"] = (_mi >= dense["First_Pos_Idx"]).astype(int)
dense["Is_Pre_Last_Positive"] = (_mi <= dense["Last_Pos_Idx"]).astype(int)
dense["Is_In_Active_Lifecycle"] = (dense["Is_Post_First_Positive"] & dense["Is_Pre_Last_Positive"]).astype(int)

dense_monthly_panel = (
    dense.drop(columns=["First_Pos_Idx", "Last_Pos_Idx"])
    .sort_values(KEY + ["Date"]).reset_index(drop=True)
)
dense_monthly_panel.to_csv(OUTPUT_MONTHLY_CSV, index=False)

def excel_safe(df):
    out = df.copy()
    for c in out.columns:
        if isinstance(out[c].dtype, (pd.PeriodDtype, pd.CategoricalDtype)):
            out[c] = out[c].astype(str)
    return out

sheets_to_export = {
    "1_Type_Mismatches": df_type_mismatch,
    "1_Column_Profile": df_col_profile,
    "1_Master_Summary": master_summary,
    "1_Grain_Uniqueness": df_grain_uniqueness,
    "1_Volume_By_Year": df_vol_year,
    "1_Volume_By_Month": df_vol_month,
    "2_Missing_Summary": df_missing_col,
    "2_Missing_By_Year": df_miss_by_year_cnt.reset_index(),
    "3_1_All_Mappings": df_mapping_summary,
}
for k, v in individual_mapping_tables.items():
    sheets_to_export[k] = v

sheets_to_export.update({
    "3_2_Map_Seg_Code": df_map_seg_anom,
    "3_2_Map_Site_Reg": df_map_site_reg_anom,
    "3_2_Map_Part_Seg": df_map_part_seg_anom,
    "3_2_Map_Part_Bulk": df_map_part_bulk_anom,
    "3_3_Orphan_Patterns": df_orphan_patterns,
    "4_1_Coverage_LowRec": df_coverage_summary,
    "4_1b_Intro_2025_LowRec": df_intro_2025,
    "4_1c_Discontinued": df_discontinued,
    "4_1d_Disc_Reg_x_Seg": df_disc_reg_seg,
    "4_2_Cross_Reg_x_Yr": cross_reg_yr.reset_index(),
    "4_2_Cross_Seg_x_Yr": cross_seg_yr.reset_index(),
    "4_2_Cross_Seg_x_Qtr": cross_seg_qtr.reset_index(),
    "4_2_Cross_Part_x_Yr": cross_part_yr.reset_index(),
    "4_2_Yr_x_Qtr_Matrix": cross_yr_qtr.reset_index(),
    "4_3_Outlier_Overview": outlier_overview,
    "4_3_Outliers_PSRS": outlier_hotspots.head(200),
    "4_3_Outliers_PartReg": outlier_hotspots_pr.head(200),
    "5_1_Cleaning_Waterfall": df_waterfall,
    "5_2_Filter_Overlap": df_overlap,
    "6_1_EDA_By_Region": eda_region,
    "6_2_EDA_By_Segment": eda_segment,
    "6_3_Parts_Seg_x_Reg": ct_parts_reg_seg.reset_index(),
    "6_4_Qty_Seg_x_Reg": ct_qty_reg_seg.reset_index(),
    "6_5a_Monthly_Seasonality": seasonality_qty.reset_index(),
    "6_5b_Quarterly_YoY_Comp": qtr_yoy_comp.reset_index(),
    "6_6_ABC_Summary": abc_summary,
    "7_1_YearIters_RowPres": report_all_15_iters,
    "7_1b_YearIters_Pos": report_pos_iters,
    "7_2_3YrIters_Pos": report_3yr_iters,
    "7_2b_3YrIters_RowPres": report_3yr_rows,
    "7_3_Iter_By_Region": ct_iter_region.reset_index(),
    "7_4_Forecasting_Tiers": tier_summary,
    "7_5_SBC_Patterns": sbc_summary,
    "7_7_Launch_Cohorts_Qtr": launch_cohorts,
    "7_7c_Lifecycle_Status": lifecycle_summary,
    "7_8_Monthly_Outliers": monthly_outlier_summary,
    "7_8_Monthly_SpikeRows": monthly_outlier_rows.head(500),
    "7_9_BulkReg_WMAPE_Check": bulk_wmape_summary,
    "7_Full_Series_Catalog": series_catalog,
})
for k, v in recon_tables.items():
    sheets_to_export[f"4_2_Recon_{k}"] = v

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    for name, tbl in sheets_to_export.items():
        excel_safe(tbl).to_excel(writer, sheet_name=name[:31], index=False)

print(f"Exported workbook -> '{OUTPUT_EXCEL}' ({len(sheets_to_export)} sheets)")
print(f"Exported dense monthly panel ({len(dense_monthly_panel):,} rows) -> '{OUTPUT_MONTHLY_CSV}'")
print(f"Saved charts -> '{OUTPUT_SEC6_PNG}' & '{OUTPUT_DASHBOARD_PNG}'")

## Cell 11: Section 10 — Export Summary Tables & Charts to PDF Report

In [ ]:
# ==========================================
# 10. EXPORT SUMMARY REPORT TO PDF (WITH FILE-LOCK FALLBACK)
# ==========================================
from matplotlib.backends.backend_pdf import PdfPages

MAX_ROWS_PDF = 30
MAX_CELL_CHARS = 50

PDF_SECTIONS = [
    ("1.1 Data Type Mismatch Report", df_type_mismatch),
    ("1.2 Column Profile (Raw)", df_col_profile),
    ("1.3 Master Entity Summary", master_summary),
    ("1.5 & 1.6 Grain & Key Uniqueness", df_grain_uniqueness),
    ("1.7 Data Volume by Year", df_vol_year),
    ("2.1 Column-Wise Missing Values", df_missing_col),
    ("3.1 Master Relationship Cardinality (All Pairs)", df_mapping_summary),
    ("3.3 Orphan Record Patterns", df_orphan_patterns),
    ("4.1 Coverage & Low-Record Summary (1, 2, <3)", df_coverage_summary),
    ("4.1b Introduced in 2025 Low-Record Breakdown", df_intro_2025),
    ("4.1d Discontinued (Part + Region) by Region x Segment", df_disc_reg_seg),
    ("4.2 Reconciliation by Region", recon_tables["Region"]),
    ("4.2 Reconciliation by Business Segment", recon_tables["Segment"]),
    ("4.2 Reconciliation by Quarter (Year-Wise)", recon_tables["Quarter_YearWise"]),
    ("4.2 Cross Reconciliation: Region x Year (Quantity)", cross_reg_yr.reset_index()),
    ("4.2 Cross Reconciliation: Business Segment x Year (Quantity)", cross_seg_yr.reset_index()),
    ("4.2 Cross Reconciliation: Business Segment x Quarter (Quantity)", cross_seg_qtr.reset_index()),
    ("4.2 Cross Reconciliation: Top 30 Part Name x Year (Quantity)", cross_part_yr.reset_index()),
    ("4.3 Transaction Outlier Overview", outlier_overview),
    ("4.3 Top Transaction Outlier Hotspots (Part-Site-Region-Segment)", outlier_hotspots),
    ("5.1 Data Cleaning Waterfall", df_waterfall),
    ("6.1 Post-Cleaning EDA by Region Id", eda_region),
    ("6.2 Post-Cleaning EDA by Business Segment", eda_segment),
    ("6.3 Distinct Part Names: Segment x Region", ct_parts_reg_seg.reset_index()),
    ("6.4 Total Quantity: Segment x Region", ct_qty_reg_seg.reset_index()),
    ("6.5a Monthly Quantity Seasonality", seasonality_qty.reset_index()),
    ("6.5b Quarterly Quantity Comparison by Year (Post-Cleaning)", qtr_yoy_comp.reset_index()),
    ("6.6 ABC / Pareto Summary", abc_summary),
    ("7.1b All 15 Year Combinations (Positive Demand Basis)", report_pos_iters),
    ("7.2 Collapsed 2023-2025 Iterations (Positive Demand)", report_3yr_iters),
    ("7.2b Collapsed 2023-2025 Iterations (Row-Presence Reference)", report_3yr_rows),
    ("7.3 Year Iteration Count by Region", ct_iter_region.reset_index()),
    ("7.4 Forecasting Eligibility Tiers", tier_summary),
    ("7.5 Demand Patterns (SBC Classification)", sbc_summary),
    ("7.7 Launch Cohorts (First Positive Year & Quarter)", launch_cohorts),
    ("7.7c Lifecycle Status Summary", lifecycle_summary),
    ("7.8 Monthly-Grain Outlier Summary", monthly_outlier_summary),
    ("7.8 Top Unexplained Monthly Outliers (Review Only)", monthly_outlier_rows),
    ("7.9 Bulk Value + Region Id WMAPE Evaluation Check", bulk_wmape_summary),
]

def fmt_pdf_table(df):
    out = excel_safe(df).copy()
    tot_r = len(out)
    trunc = tot_r > MAX_ROWS_PDF
    if trunc:
        out = out.head(MAX_ROWS_PDF).copy()
    for col in out.columns:
        if pd.api.types.is_float_dtype(out[col]):
            out[col] = out[col].map(lambda v: "" if pd.isna(v) else (f"{int(v):,}" if v == int(v) and abs(v) >= 1000 else f"{v:,.2f}"))
        elif pd.api.types.is_integer_dtype(out[col]):
            out[col] = out[col].map(lambda v: "" if pd.isna(v) else f"{int(v):,}")
        else:
            out[col] = out[col].fillna("").astype(str).map(lambda s: s if len(s) <= MAX_CELL_CHARS else s[:MAX_CELL_CHARS - 3] + "...")
    out.columns = [str(c).replace("_", " ") for c in out.columns]
    return out, tot_r, trunc

def write_pdf(target_path):
    with PdfPages(target_path) as pdf:
        for img_path in [OUTPUT_SEC6_PNG, OUTPUT_DASHBOARD_PNG]:
            if os.path.exists(img_path):
                fig, ax = plt.subplots(figsize=(11, 8.5))
                ax.axis("off")
                ax.imshow(plt.imread(img_path))
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

        for title, raw_tbl in PDF_SECTIONS:
            if raw_tbl is None or len(raw_tbl) == 0:
                continue
            fmt_tbl, tot_r, trunc = fmt_pdf_table(raw_tbl)
            fig, ax = plt.subplots(figsize=(11, 8.5))
            ax.axis("off")
            sub = f" (Top {MAX_ROWS_PDF} of {tot_r:,} rows)" if trunc else ""
            ax.set_title(title + sub, fontsize=11, fontweight="bold", pad=12)
            t = ax.table(cellText=fmt_tbl.values, colLabels=fmt_tbl.columns, loc="center", cellLoc="left")
            t.auto_set_font_size(False)
            t.set_fontsize(max(5.5, min(8.0, 68.0 / max(len(fmt_tbl.columns), 1))))
            t.scale(1, 1.25)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

try:
    write_pdf(OUTPUT_PDF)
    print(f"Exported PDF report -> '{OUTPUT_PDF}'")
except PermissionError:
    alt_pdf = OUTPUT_PDF.replace(".pdf", f"_{datetime.now():%Y%m%d_%H%M%S}.pdf")
    write_pdf(alt_pdf)
    print(f"'{OUTPUT_PDF}' was open/locked; exported instead -> '{alt_pdf}'")